In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# ---- CONFIG ----
TRIP_DATA_FILE = "Taxi_Trips__2024-__20250611.csv"
NUM_VEHICLES = 1000
START_TIME = datetime(2025, 4, 1, 17, 0, 0)
END_TIME = datetime(2025, 4, 2, 0, 0, 0)

# ---- LOAD TRIP DATA ----
trip_df = pd.read_csv(TRIP_DATA_FILE)  # Assuming tab-separated input
trip_df = trip_df.dropna(subset=["Pickup_Centroid_Latitude", "Pickup_Centroid_Longitude"])
trip_df = trip_df.reset_index(drop=True)

# ---- SAMPLE POINTS ----
step = len(trip_df) // NUM_VEHICLES
sampled = trip_df.iloc[::step][["Pickup_Centroid_Latitude", "Pickup_Centroid_Longitude"]].reset_index(drop=True)

# ---- JITTER FUNCTION ----
def jitter_coordinates(lat, lon, max_offset=0.0045):  # ~500m
    lat_offset = np.random.uniform(-max_offset, max_offset)
    lon_offset = np.random.uniform(-max_offset, max_offset)
    return lat + lat_offset, lon + lon_offset

# ---- VEHICLE TYPE GENERATION ----
def generate_vehicle_type(index):
    if index < 0.4 * NUM_VEHICLES:
        return "EV"
    elif index < 0.7 * NUM_VEHICLES:
        return "Diesel"
    else:
        return "Petrol"

# ---- TIME GENERATOR ----
def generate_random_time():
    delta = END_TIME - START_TIME
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return START_TIME + timedelta(seconds=random_seconds)

# ---- BUILD VEHICLE DATA ----
vehicle_rows = []

for i in range(NUM_VEHICLES):
    lat, lon = jitter_coordinates(sampled.loc[i, "Pickup_Centroid_Latitude"], sampled.loc[i, "Pickup_Centroid_Longitude"])
    vehicle_type = generate_vehicle_type(i)
    battery = np.random.randint(20, 101) if vehicle_type == "EV" else None
    last_time = generate_random_time()

    vehicle_rows.append({
        "Vehicle_ID": f"VEH{i:05d}",
        "Type": vehicle_type,
        "Current_Latitude": lat,
        "Current_Longitude": lon,
        "Battery_Percentage": battery,
        "Status": "Idle",
        "Last_Assigned_Time": last_time.strftime("%Y-%m-%d %H:%M:%S")
    })

vehicle_df = pd.DataFrame(vehicle_rows)

# ---- SAVE ----
vehicle_df.to_csv("generated_vehicle_data.csv", index=False)
print(vehicle_df.sample(n=10))


     Vehicle_ID    Type  Current_Latitude  Current_Longitude  \
4679   VEH04679  Diesel         41.782615         -87.753671   
296    VEH00296      EV         41.877153         -87.638163   
3705   VEH03705      EV         41.877663         -87.620817   
7517   VEH07517  Petrol         41.854296         -87.617524   
2462   VEH02462      EV         41.824307         -87.596623   
117    VEH00117      EV         41.885296         -87.636540   
7292   VEH07292  Petrol         41.875247         -87.670916   
1948   VEH01948      EV         41.897899         -87.620410   
6184   VEH06184  Diesel         41.902731         -87.634866   
6882   VEH06882  Diesel         41.898415         -87.630862   

      Battery_Percentage Status   Last_Assigned_Time  
4679                 NaN   Idle  2025-04-01 21:57:29  
296                 75.0   Idle  2025-04-01 20:07:42  
3705                95.0   Idle  2025-04-01 21:59:21  
7517                 NaN   Idle  2025-04-01 21:46:15  
2462                